In [0]:
import os,sys

os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")

print("Thread limits set.")

Thread limits set.


In [0]:
## Parameters (widgets)
dbutils.widgets.text("only_node", "", "Node to train")
only_node = dbutils.widgets.get("only_node").strip()
if only_node:
    os.environ['MCH_ONLY_NODE']=only_node
print('Training node:', only_node)


Training node: 


In [0]:
import os, sys, time, json, math, traceback
from typing import Any, Dict, Optional
from pathlib import Path

import joblib
import mlflow
import mlflow.sklearn

sys.path.append('/Workspace/9900-f18a-cake/working_branch/src')
from mch.models.training import BatchModelTrainer

print("Imports loaded.")

[R INIT] limma loaded successfully via rpy2
Imports loaded.


In [0]:
def _norm_exp(v: str | None) -> str | None:
    if v is None:
        return None
    v = str(v).strip()
    if v == "" or v.lower() == "none":
        return None
    return v

def _set_experiment() -> str:
    """
    read env paramter as expperment path:
      - MLFLOW_EXPERIMENT_NAME / MLFLOW_EXPERIMENT_PATH
    Or Falling back to Databricks Notebook path as Experiment Name.
    """
    exp = (
        _norm_exp(os.getenv("MLFLOW_EXPERIMENT_NAME"))
        or _norm_exp(os.getenv("MLFLOW_EXPERIMENT_PATH"))
    )
    if exp is None:
        print("Falling back to Databricks Notebook path as Experiment Name.")
        return None
    mlflow.set_experiment(exp)
    print("Using experiment:", exp)
    return exp

def _is_scalar(x: Any) -> bool:
    return isinstance(x, (int, float)) and not isinstance(x, bool)

def _safe_node_name(nid: str) -> str:
    return "".join(c if c.isalnum() else "_" for c in nid)

def _log_metrics_from_node(node_id: str, node_stats: Dict[str, Any]) -> Dict[str, float]:
   
    logged: Dict[str, float] = {}

    metrics = node_stats.get("metrics")
    if isinstance(metrics, dict):
        items = metrics.items()
    else:
        items = node_stats.items()

    for k, v in items:
        if not _is_scalar(v):
            continue
        key = f"{node_id}.{k}"
        logged[key] = float(v)

    if logged:
        mlflow.log_metrics(logged)

    return logged

def _log_model_for_node(
    node_id: str, 
    node_stats: Dict[str, Any], 
    save_mode: str,
    trainer: Optional['BatchModelTrainer'] = None 
) -> None:

    save_mode = (save_mode or "nosave").lower()
    if save_mode == "nosave":
        print(f"[SAVE] Skip saving for {node_id} (MCH_SAVE_MODE=nosave)")
        return

    input_example = node_stats.get("input_example")

    est = (
        node_stats.get("estimator")
        or node_stats.get("model")
    )
    if est is None and trainer is not None:
        est = trainer.models.get(node_id)
        if est is not None:
            print(f"[SAVE] Found estimator in trainer.models for {node_id} (HACK)")

    if est is None:
        print(f"[SAVE] No estimator in stats/trainer.models for {node_id}, skip.")
        return

    safe_id = _safe_node_name(node_id)
    
    base_dir = Path("/tmp") / safe_id
    base_dir.mkdir(parents=True, exist_ok=True)

    stats_to_save = dict(node_stats)
    stats_to_save.pop("estimator", None)
    stats_to_save.pop("model", None)

    stats_path = base_dir / f"{safe_id}_stats.json"
    with open(stats_path, "w", encoding="utf-8") as f:
        json.dump(stats_to_save, f, indent=2, default=str)

    if save_mode in ("joblib-only", "all", "mlflow-only"):
        model_path = base_dir / f"{safe_id}_model.joblib"
        try:
            joblib.dump(est, model_path)
        except Exception as e:
            print(f"[SAVE] joblib dump failed for {node_id}: {e}")
            model_path = None

        artifact_subdir = f"{node_id}/joblib"
        mlflow.log_artifact(str(stats_path), artifact_path=artifact_subdir)
        if model_path and model_path.exists():
            mlflow.log_artifact(str(model_path), artifact_path=artifact_subdir)

    if save_mode in ("mlflow-only", "all"):
        try:
            mlflow.sklearn.log_model(
                sk_model=est,
                artifact_path=f"models/{safe_id}",
                input_example=input_example,
            )
        except Exception as e:
            print(f"[SAVE] mlflow.sklearn.log_model failed for {node_id}: {e}")

def _print_summary(result: Dict[str, Any]) -> None:
    print("\n" + "=" * 60)
    print("Child training summary")
    print("ok          :", result.get("ok"))
    print("node_id     :", result.get("node_id"))
    print("elapsed_sec :", result.get("t_sec"))
    if result.get("metrics"):
        print("metrics:")
        for k, v in result["metrics"].items():
            print(f"  - {k}: {v}")
    if result.get("error"):
        print("error       :", result["error"])
    try:
        run = mlflow.active_run()
        if run:
            print("mlflow_run  :", run.info.run_id)
    except Exception:
        pass
    print("=" * 60 + "\n")


In [0]:

os.environ.setdefault("MCH_SAVE_MODE", "all")
os.environ["MCH_DISABLE_DM"]        = "0"
EXP_NAME = _set_experiment()

PARENT_RUN_ID = (
    os.getenv("PARENT_RUN_ID")
    or os.getenv("MLFLOW_PARENT_RUN_ID")
    or os.getenv("DATABRICKS_PARENT_RUN_ID")
)

save_mode = os.getenv("MCH_SAVE_MODE", "nosave").lower()
print("Experiment :", EXP_NAME)
print("Parent run :", PARENT_RUN_ID)
print("Only node  :", only_node or "(all)")
print("Save mode  :", save_mode)

result: Dict[str, Any] = {
    "ok": False,
    "node_id": only_node or "(all)",
    "metrics": {},
    "error": None,
    "trace": None,
}

t0 = time.time()
stats_all: Dict[str, Any] = {}

try:
    with mlflow.start_run(
        run_name=f"child-{only_node or 'all'}",
        nested=bool(PARENT_RUN_ID),
    ) as run:
        if PARENT_RUN_ID:
            mlflow.set_tag("mlflow.parentRunId", PARENT_RUN_ID)

        mlflow.set_tags({
            "NODE_ID": only_node or "(all)",
            "MCH_SAVE_MODE": save_mode,
        })

        trainer = BatchModelTrainer()
        stats = trainer.train_all_models(
            raise_on_error=False,
        )

        if isinstance(stats, dict):
            stats_all = stats
        else:
            stats_all = {only_node or "root": stats}

        mlflow.log_dict(stats_all, "stats.json")

        for nid, node_stats in stats_all.items():
            nid_str = str(nid)
            print(f"=== logging node: {nid_str} ===")
            m = _log_metrics_from_node(nid_str, node_stats)
            result["metrics"].update(m)
            _log_model_for_node(nid_str, node_stats, save_mode, trainer)

        result["ok"] = True

except Exception as e:
    result["error"] = f"{type(e).__name__}: {e}"
    result["trace"] = traceback.format_exc()
    print(result["trace"])

finally:
    result["t_sec"] = round(time.time() - t0, 3)
    _print_summary(result)

print("Finished")
display(stats_all)
# dbutils.notebook.exit(json.dumps(result))


Falling back to Databricks Notebook path as Experiment Name.
Experiment : None
Parent run : None
Only node  : Haematological malignancy
Save mode  : all


2025-11-25 19:46:03,959 INFO mch.training: Only training specified node: Haematological malignancy
2025-11-25 19:46:03,960 INFO mch.training: Training model for node: Haematological malignancy


Haematological malignancy origin data count: 2470
Haematological malignancy mvalue_df columns: ['biosample_id', 'cg00000029', 'cg00000109', 'cg00000155', 'cg00000158']
Haematological malignancy diseaseTree count: 496
Haematological malignancy joint count: 482
Haematological malignancy joint samples: ['1UHMQ42P_T_16ATZUQY_M', '1Z5PB65G_T_BRC2KNQ9_M', '1L17C7ZA_T_HS6MGVUH_M', '2ZYVUCYK_T_U5G1BCGL_M', '6XTK90GT_T_19GK2A6S_M']
Haematological malignancy Data after filter: 482
Haematological malignancy Data after design filter: 482
Haematological malignancy 482 482
design type： <class 'polars.dataframe.frame.DataFrame'>


2025-11-25 19:46:29,542 INFO mch.training: Prefilter start: input shape=(482,600118), candidates=600117, topk=0, scan_max=20000, chunk_size=5000
2025-11-25 19:46:29,545 INFO mch.training: Prefilter bypass: kept=600117
2025-11-25 19:46:59,935 INFO mch.training: Train/Test shapes: X_train=(385, 600117), X_test=(97, 600117)
2025-11-25 19:46:59,936 INFO mch.training: DM enabled
2025-11-25 19:46:59,938 INFO mch.training: Pipeline steps: ['replace_inf', 'differentialMethylation', 'modelGeneration']


Fitting 3 folds for each of 1 candidates, totalling 3 fits
fitting via limma
probe identification for: Leukaemia, 1 of 4 cancer types
Running analysis


com.databricks.backend.common.rpc.CommandCancelledException
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$5(SequenceExecutionState.scala:132)
	at scala.Option.getOrElse(Option.scala:189)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3(SequenceExecutionState.scala:132)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3$adapted(SequenceExecutionState.scala:129)
	at scala.collection.immutable.Range.foreach(Range.scala:158)
	at com.databricks.spark.chauffeur.SequenceExecutionState.cancel(SequenceExecutionState.scala:129)
	at com.databricks.spark.chauffeur.ExecContextState.cancelRunningSequence(ExecContextState.scala:715)
	at com.databricks.spark.chauffeur.ExecContextState.$anonfun$cancel$1(ExecContextState.scala:435)
	at scala.Option.getOrElse(Option.scala:189)
	at com.databricks.spark.chauffeur.ExecContextState.cancel(ExecContextState.scala:435)
	at com.databricks.spark.chauffeur.ExecutionContextManagerV1.can